In [1]:
from ultralytics import YOLO
import os

In [2]:
model = YOLO(os.path.join("runs","detect","train-3","weights","best.pt"))

In [ ]:
images_val = os.path.join("training_data","images","val")
labels_val = os.path.join("training_data","labels","val")
os.listdir(images_val)

In [ ]:
os.listdir(labels_val)

In [ ]:
sample = "105e0c65-BikesHelmets81"

class_labels = ["With Helmet", "Without Helmet"]
with open(os.path.join(labels_val, sample+'.txt')) as f:
    int_cls = int(f.read().split(" ")[0])
    print(class_labels[int_cls])

from IPython.display import Image, display
display(Image(filename = os.path.join(images_val, sample+'.png')))

In [6]:
model = YOLO(os.path.join("runs","detect","train-3","weights","best.pt"))

In [ ]:
img_path = os.path.join(images_val, sample+'.png')
y_pred = model.predict(img_path, conf = 0.4)


In [ ]:
for result in y_pred:
    if not result.boxes:
        print("No predictions")
        continue

    for box in result.boxes:
        cls_id = int(box.cls[0].item())
        detected = result.names[cls_id]

        confidence = box.conf[0].item()

        xmin, ymin, xmax, ymax = box.xyxy[0].tolist()

        print(f"Detected: {detected} | confidence: {confidence:.2f} | Box co-ords: {xmin:.2f}, {ymin:.2f}, {xmax:.2f}, {ymax:.2f}")

In [ ]:
%pip install opencv-python

In [ ]:
import cv2


# Load trained model
# Use the relative path to where the file is actually located
model = YOLO("runs/detect/train-3/weights/best.pt")  # Replace with your model path

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

while True:
    success, frame = cap.read()

    if not success:
        print("Error: Could not read frame.")
        break

    # Run inference
    results = model.predict(frame, conf=0.4)

    for result in results:
        if len(result.boxes) == 0:
            continue

        for box in result.boxes:
            # Get class information
            cls_id = int(box.cls[0].item())
            detected = result.names[cls_id]
            confidence = box.conf[0].item()

            # Get bounding box coordinates
            xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())

            # Set color based on class
            if detected == "With Helmet":
                color = (0, 255, 0)      # Green
            elif detected == "Without Helmet":
                color = (0, 0, 255)      # Red
            else:
                color = (255, 255, 255)  # White (fallback)

            # Draw bounding box
            cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 2)

            # Draw label
            label = f"{detected} {confidence:.2f}"
            cv2.putText(
                frame,
                label,
                (xmin, ymin - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    # Display output
    cv2.imshow("Helmet Detection", frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()